# 03 - Training a CNN to Detect RFI

**Goal of this notebook:** train a small convolutional neural network
(CNN) to predict the AOFlagger RFI mask directly from an amplitude patch,
using the dataset created in `02_create_dataset.ipynb`.

This is **Part 2** of the workshop. It picks up exactly where Part 1 left
off: a `dataset/` folder containing `train/`, `validation/`, `test/`
splits of `.npy` image/label patches, plus a `metadata.csv` describing
every patch.

You will:
1. Load the dataset with a PyTorch `Dataset`/`DataLoader`.
2. Look at the model architecture (a small U-Net).
3. Train the model for a few epochs, watching the training/validation loss and F1 score.
4. Evaluate the trained model on the held-out test set.
5. Visually inspect some predictions.

> **Note:** this notebook requires PyTorch (`pip install torch`), which
> is only needed for Part 2 - Part 1 (dataset creation) does not depend
> on it at all.


## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import torch

from models.patch_dataset import RFIPatchDataset
from models.cnn import SimpleRFIUNet, count_trainable_parameters
from models.train import TrainConfig, train
from models.evaluate import load_trained_model, evaluate_on_test_set, visualize_predictions, print_metrics_report

%matplotlib inline


## 2. Load the configuration

All training hyperparameters (epochs, batch size, learning rate, loss
function, model width, ...) live in `configs/train.yaml` so you can
experiment without editing any code.


In [ ]:
config = TrainConfig.from_yaml("../configs/train.yaml")
print(config)
print("Device:", config.resolve_device())


## 3. Look at the data

Let's load the training split and look at one example, to double check
image/label shapes line up with what the model expects.


In [ ]:
train_dataset = RFIPatchDataset(config.dataset_dir, split="train")
val_dataset = RFIPatchDataset(config.dataset_dir, split="validation")
test_dataset = RFIPatchDataset(config.dataset_dir, split="test")

print(f"Train patches:      {len(train_dataset)}")
print(f"Validation patches: {len(val_dataset)}")
print(f"Test patches:       {len(test_dataset)}")

image, label = train_dataset[0]
print(f"Image tensor shape: {image.shape} (channels, height, width)")
print(f"Label tensor shape: {label.shape}")


## 4. The model: a small U-Net

RFI detection here is a *pixel-level segmentation* problem: given an
amplitude patch, predict a same-sized mask of clean (0) vs RFI (1)
pixels. A U-Net is a natural fit: an encoder that downsamples while
extracting features, and a decoder that upsamples back to full
resolution, with "skip connections" that preserve fine spatial detail.


In [ ]:
model = SimpleRFIUNet(in_channels=1, base_channels=config.base_channels)
print(model)
print(f"Trainable parameters: {count_trainable_parameters(model):,}")


## 5. Train the model

This runs the full training loop from `models/train.py`: for each epoch,
it trains on the training set and evaluates on the validation set,
saving the model with the best validation F1 score to
`checkpoints/best_model.pt`.

Feel free to lower `config.epochs` for a quick smoke test first.


In [ ]:
history = train(config)


### Plot the training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss over training")
axes[0].legend()

axes[1].plot(history["train_f1"], label="train")
axes[1].plot(history["val_f1"], label="validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 score")
axes[1].set_title("F1 score over training")
axes[1].legend()

fig.tight_layout()
plt.show()


## 6. Evaluate on the held-out test set

Now let's reload the *best* checkpoint (by validation F1) and see how it
performs on data it has never seen at all - neither for training nor for
model selection.


In [ ]:
best_model = load_trained_model(f"{config.checkpoint_dir}/best_model.pt")
test_metrics = evaluate_on_test_set(best_model, config.dataset_dir)
print_metrics_report(test_metrics)


## 7. Visualize predictions

Numbers only tell part of the story - let's look at some actual
predictions: the input amplitude patch, the AOFlagger ground truth, the
model's predicted mask, and an overlay of the prediction on the input.


In [ ]:
fig = visualize_predictions(best_model, config.dataset_dir, split="test", n_samples=4)
plt.show()


## 8. Things to try

- Switch `loss_function` between `bce` and `dice` in `configs/train.yaml` - which converges faster? Which gives a better test F1/IoU?
- Increase `base_channels` (e.g. 16 -> 32) - does it improve results? How does training time change?
- Look at the *worst* predictions (lowest per-patch F1) rather than random ones - what do they have in common?
- Try training on patches produced with a different `normalization` method from Part 1 - does it change how well the CNN learns?
- This is a simple 2-level U-Net. Can you extend `models/cnn.py` to add a third encoder/decoder stage?

This concludes the two-part workshop: you've gone from a raw MeerKAT
`.rdb` file all the way to a trained CNN that can flag RFI in radio
astronomy data!
